# CRAda Single-Slice Benchmark

In [ ]:
import sys
from pathlib import Path
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import torch

sys.path.append("//wsl.localhost/Ubuntu/home/csrao/git-personal/llmr")  # Workstation
from llmr.conversion import torch2np_clean
from llmr.fft import fft2c, ifft2c
from llmr.metrics import psnr, ssim

sys.path.append("c:\\Users\\csrao\\git-personal\\pnp-cosmo")
from cosmo.utils import load_config
from cosmo.cosmo_systems import StochasticContentMUNIT
from data.nyudicom_t1wt2w_recon_dataset import NYUDicomT1WT2WReconVolumeDataset
from recon.algorithms import l1_wavelet_ista, pnp_cosmo

In [ ]:
DATA_ROOT = Path("D:/Datasets/NYU_fastMRI/brain_dicom_t1t2_dataset")
COSMO_CONFIG_PATH = Path("../configs/nyu_pft.yaml")
COSMO_CHECKPOINT_PATH = Path("...")
SEED = 0

# Reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
def load_cosmo(config_path, checkpoint_path):
    conf = load_config(config_path)
    cosmo = StochasticContentMUNIT(conf)
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    for k in ['autoenc_1', 'autoenc_2']:
        cosmo.networks[k].load_state_dict(checkpoint[f'net_{k}'])
        cosmo.networks[k].eval()
    return cosmo

def estimate_phase_map(kspace):
    cf = 0.08
    xshape, yshape = kspace.shape[2], kspace.shape[3]
    xnlf, ynlf = round(cf * xshape), round(cf * yshape)
    lowres_mask = torch.zeros_like(kspace)
    lowres_mask[:, :, xshape//2-xnlf//2 : xshape//2+xnlf//2, yshape//2-ynlf//2 : yshape//2+ynlf//2] = 1.
    lowres_kspace = kspace * lowres_mask
    lowres_image = ifft2c(lowres_kspace)
    phase_map_estim = lowres_image.angle()
    return phase_map_estim

def compute_metrics(image_gt, image_estim, recon_intensity_range):
    image_gt = torch2np_clean(image_gt.abs())
    image_estim = torch2np_clean(image_estim.abs())
    ssim_value = ssim(image_gt, image_estim, data_range=recon_intensity_range)
    psnr_value = psnr(image_gt, image_estim, data_range=recon_intensity_range)
    mse_value = np.mean((image_gt - image_estim) ** 2)
    return ssim_value, psnr_value, mse_value

def show_recon(image_estim, image_gt, recon_intensity_range):
    image_estim = image_estim.cpu().squeeze()
    image_gt = image_gt.cpu().squeeze()
    errormap_recon_gt = (image_estim.abs() - image_gt.abs())
    fig, axs = plt.subplots(1, 2, figsize=(7, 5))
    axs[0].imshow(image_estim.abs(), cmap='gray', vmin=0, vmax=recon_intensity_range)
    axs[1].imshow(errormap_recon_gt.squeeze(), cmap='coolwarm', vmin=-0.1*recon_intensity_range, vmax=0.1*recon_intensity_range)
    [ax.axis('off') for ax in axs]
    fig.tight_layout()
    plt.subplots_adjust(wspace=0, hspace=0)
    plt.show()

In [ ]:
volume_idx, slice_idx = 0, 0

dataset = NYUDicomT1WT2WReconVolumeDataset(DATA_ROOT, fold=1, recon_contrast='t2w', accel=4, center_frac=0.08, kspace_noise_factor=0.01)
volume_data = dataset[volume_idx]
slice_data = {k: volume_data[k][slice_idx].unsqueeze(0) for k in ['image_ref', 'image_gt', 'kspace', 'mask']}

fig, axs = plt.subplots(1, 3, figsize=(6,5))
axs[0].imshow(slice_data['image_ref'].squeeze(), cmap='gray')
axs[1].imshow(slice_data['image_gt'].squeeze(), cmap='gray')
axs[2].imshow(slice_data['mask'].squeeze(), cmap='gray')
[ax.axis('off') for ax in axs.ravel()]
fig.tight_layout()
plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

# Prepare inputs
kspace = slice_data['kspace'].unsqueeze(0).to('cuda')
mask = slice_data['mask'].unsqueeze(0).to('cuda')
ground_truth = slice_data['image_gt'].unsqueeze(0).to('cuda')
image_ref = slice_data['image_ref'].unsqueeze(0).to('cuda')

# Estimate unknowns
recon_intensity_range = float(volume_data['image_gt'].max())
ref_intensity_range = float(volume_data['image_ref'].max())
phase_map_estim = estimate_phase_map(kspace)
max_eig = 1.

# Load CoSMo
cosmo = load_cosmo(COSMO_CONFIG_PATH, COSMO_CHECKPOINT_PATH)

### Recon

In [ ]:
input = {
    'kspace': kspace, 
    'mask': mask, 
    'csm': torch.ones_like(kspace),
    'max_eig': 1.,
    'ground_truth': ground_truth,  # To compute error curves
    }

# Baseline: L1-wavelet
config = {'weight': 0.1, 'num_iters': 100}
output = l1_wavelet_ista(input, config)
show_recon(output['image_estim'], ground_truth, recon_intensity_range)
ssim_value, psnr_value, mse_value = compute_metrics(ground_truth, output['image_estim'], recon_intensity_range)
print(ssim_value, psnr_value, mse_value)

# PnP-CoSMo
input_copy = deepcopy(input)
input_copy.update({'cosmo': cosmo, 'recon_intensity_range': recon_intensity_range, 'ref_intensity_range': ref_intensity_range, 'image_ref': image_ref, 'phase_map': phase_map_estim, 'ref_domain': 't1w', 'recon_domain':'t2w'})
config = {'num_iters': 1, 'ideal_content': False, 'cr_enable': False, 'cr_step_size': 1e0}
output = pnp_cosmo(input_copy, config)
show_recon(output['image_estim'], ground_truth, recon_intensity_range)
fig, axs = plt.subplots(1,3,figsize=(16,3))
axs[0].plot(output['content_nmse']); axs[0].set_title('content NMSE')
axs[1].plot(output['style_nmse']); axs[1].set_title('style NMSE')
axs[2].plot(output['recon_nmse']); axs[2].set_title('recon NMSE')
plt.show()
ssim_value, psnr_value, mse_value = compute_metrics(ground_truth, output['image_estim'], recon_intensity_range)
print(ssim_value, psnr_value, mse_value)